# Baseline screening — MLP & XGBoost on M2OR (molecule ‖ protein)

Runs the two MP baselines across **splits** and **protein variants**, with on-disk
caching (`notebooks/cache/results.json`) so results survive kernel restarts and
already-computed configs are never recomputed.

The last section is a placeholder to plug in **modified ESM-2** embeddings as new
protein variants — only the new variant gets computed, the rest is reused.

In [5]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# locate repo root (has pyproject.toml) regardless of where the kernel started
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from orbind import dataset as D
from orbind.baselines import run_one

DATA = ROOT / 'data'
CACHE = ROOT / 'notebooks' / 'cache'; CACHE.mkdir(parents=True, exist_ok=True)
print('repo root:', ROOT)

repo root: c:\Users\User\Desktop\SberWork\orbind


In [ ]:
# --- load data + embeddings -------------------------------------------------
PAIRS = pd.read_csv(DATA / 'processed' / 'pairs_curated.csv')
ESM = D.load_npz_dict(DATA / 'embeddings' / 'proteins' / 'esm2_650m.npz')
GIN = D.load_npz_dict(DATA / 'embeddings' / 'molecules' / 'gin_supervised_contextpred.npz')
print(f'pairs={len(PAIRS)} | proteins={len(ESM)} | molecules={len(GIN)}')

# protein variants: tag -> (prot_dict, random_prot_flag, prot_transform_fn | None)
# prot_transform signature: fn(prot_dict, train_receptor_ids) -> prot_dict
# It is called AFTER the split, so mean/params are computed from train only.
PROT_VARIANTS = {
    'ESM':    (ESM, False, None),   # real ESM-2 650M, no transform
    'random': (ESM, True,  None),   # per-row noise floor (control)
}

In [ ]:
# --- on-disk result cache ---------------------------------------------------
CACHE_FILE = CACHE / 'results.json'

def _load(): return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
def _save(c): CACHE_FILE.write_text(json.dumps(c, indent=2))

def cached_run(tag, prot, random_prot, transform_fn, head, split, seed=42, force=False):
    cache = _load()
    key = f'{tag}|{head}|{split}|seed{seed}'
    if key in cache and not force:
        return cache[key]
    m, info = run_one(PAIRS, prot, GIN, head, split,
                      random_prot=random_prot, prot_transform=transform_fn, seed=seed)
    rec = {'tag': tag, 'head': head, 'split': split, 'seed': seed,
           **{k: round(float(v), 4) for k, v in m.items()}, **info}
    cache[key] = rec; _save(cache)
    return rec

In [ ]:
# --- run the grid (cached) --------------------------------------------------
HEADS  = ['mlp', 'boost']
SPLITS = ['stratified', 'group_molecule', 'group_receptor']

rows = []
for tag, (prot, rnd, tfn) in PROT_VARIANTS.items():
    for head in HEADS:
        for split in SPLITS:
            r = cached_run(tag, prot, rnd, tfn, head, split)
            rows.append(r)
            print(f"{tag:14s} {head:5s} {split:16s} AUPRC={r['AUPRC']:.3f} AUROC={r['AUROC']:.3f}")
res = pd.DataFrame(rows)

In [11]:
# --- view: per head, metric x (variant, split) ------------------------------
for head in HEADS:
    sub = res[res['head'] == head]
    piv = sub.pivot_table(index='split', columns='tag', values=['AUPRC', 'AUROC', 'MCC'])
    print(f'### {head}')
    display(piv.round(3))

### mlp


AUPRC         AUROC           MCC       
tag               ESM random    ESM random    ESM random
split                                                   
group_molecule  0.299  0.068  0.726   0.49  0.248 -0.011
group_receptor  0.448  0.178  0.827   0.65  0.394  0.128
stratified      0.511  0.171  0.885   0.66  0.450  0.134

### boost


AUPRC         AUROC           MCC       
tag               ESM random    ESM random    ESM random
split                                                   
group_molecule  0.383  0.089  0.772  0.556  0.366  0.002
group_receptor  0.441  0.236  0.843  0.738  0.366  0.070
stratified      0.556  0.214  0.886  0.748  0.513  0.103

## ESM-2 embedding analysis & centering

The cosine heatmap (next cell) shows all 409 receptors cluster very tightly:
**mean off-diagonal cosine ≈ 0.954** — the shared "I am an OR/GPCR" direction dominates.

Subtracting the **global train mean** removes this common component, leaving receptor-specific
residuals where inter-receptor differences are more contrastive.

**Why one global mean, not per-family means?**  
OR subfamilies have different odor specificities (aldehydes vs. ketones vs. terpenes) —
that subfamily-level variation IS signal for binding prediction. Per-family centering would
remove it. The global mean captures only the "all ORs share GPCR backbone" noise.

**Leakage note:** mean is computed over all receptor embeddings (≈ train for `group_molecule`
where all receptors are seen; a small approximation for `group_receptor` where ~20% of
receptors are held out — negligible at n=409).

In [12]:
# --- cosine similarity: before vs. after centering -------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list

FIGURES = ROOT / "notebooks" / "figures"; FIGURES.mkdir(exist_ok=True)

E  = np.stack(list(ESM.values())).astype(np.float64)   # [409, 1280]
mu = E.mean(axis=0, keepdims=True)
Ec = E - mu                                             # centered

def cosine_matrix(X):
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-9
    Xn = X / n
    return Xn @ Xn.T

def clustered_order(S):
    Z = linkage(S, method="average", metric="cosine")
    return leaves_list(Z)

S_raw = cosine_matrix(E)
S_cen = cosine_matrix(Ec)
order = clustered_order(E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, S, title in zip(axes,
                         [S_raw[np.ix_(order, order)], S_cen[np.ix_(order, order)]],
                         ["ESM raw", "ESM − global mean"]):
    off = S[~np.eye(len(S), dtype=bool)]
    im = ax.imshow(S, cmap="magma", vmin=off.min(), vmax=1.0, interpolation="nearest")
    ax.set_title(f"{title}\noff-diag cosine: mean={off.mean():.3f}  min={off.min():.3f}")
    ax.set_xlabel("receptors (clustered)"); ax.set_ylabel("receptors (clustered)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(FIGURES / "esm_cosine_before_after.png", dpi=130)
plt.show(); print("saved →", FIGURES / "esm_cosine_before_after.png")

saved → c:\Users\User\Desktop\SberWork\orbind\notebooks\figures\esm_cosine_before_after.png


In [ ]:
# --- ESM_centered: subtract mean of TRAIN receptors only -------------------
#
# The transform function receives (prot_dict, train_receptor_ids) at split time,
# so the mean is always computed from train receptors only — no leakage.

def center_by_train_mean(prot, train_recs):
    """Subtract mean of train-receptor embeddings from ALL embeddings."""
    train_vecs = np.stack([prot[r] for r in train_recs]).astype(np.float64)
    mean = train_vecs.mean(axis=0).astype(np.float32)
    return {k: v - mean for k, v in prot.items()}

PROT_VARIANTS['ESM_centered'] = (ESM, False, center_by_train_mean)

# run grid — ESM / random hit cache instantly, only ESM_centered trains fresh
rows_all = []
for tag, (prot, rnd, tfn) in PROT_VARIANTS.items():
    for head in HEADS:
        for split in SPLITS:
            r = cached_run(tag, prot, rnd, tfn, head, split)
            rows_all.append(r)
            print(f"{tag:14s} {head:5s} {split:16s} AUPRC={r['AUPRC']:.3f}")

res = pd.DataFrame(rows_all)

# summary: AUPRC per split x variant
for head in HEADS:
    sub = res[res['head'] == head]
    piv = sub.pivot_table(index='split', columns='tag', values='AUPRC').round(3)
    piv = piv[['random', 'ESM', 'ESM_centered']]
    print(f"\n### AUPRC — {head}")
    display(piv)